In [0]:
%run /Workspace/Users/n.moukayed@gmail.com/databricks_projects/Notebooks/NB_Collibri_Bronze

In [0]:
# Import additional libraries
from pyspark.sql import Row
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
    DateType,
)
import pytest
import sys

In [0]:
# Function 1: Create test dataframe 
def create_test_dataframe(data, schema):
    """Helper to create test DataFrames."""
    return spark.createDataFrame(data, schema)

# Function 2: Verify Columns
def assert_columns_exist(df, expected_columns):
    actual_columns = set(df.columns)
    missing_columns = set(expected_columns) - actual_columns
    assert len(missing_columns) == 0, f"Missing columns: {missing_columns}"


# Unit Tests

In [0]:
def test_read_csv_files_structure():
    """Test that read_csv_files returns DataFrame with expected columns."""
    logger.info("Running test: read_csv_files_structure")
    
    # Read actual data
    df = read_csv_files(SOURCE_PATH)
    
    # Expected columns
    expected_columns = [
        "timestamp",
        "turbine_id",
        "wind_speed",
        "wind_direction",
        "power_output",
        "_rescued_data",
        "_source_file",
        "_file_name",
        "_file_group",
        "_ingested_at",
        "_ingestion_date",
    ]
    
    # Assert columns exist
    assert_columns_exist(df, expected_columns)
    
    # Assert _file_group is extracted correctly
    file_groups = df.select("_file_group").distinct().collect()
    assert len(file_groups) > 0, "Should have file groups"
    
    logger.info("test_read_csv_files_structure passed")


def test_read_csv_files_file_group_extraction():
    """Test that file group is correctly extracted from filename."""
    logger.info("Running test: read_csv_files_file_group_extraction")
    
    # Create test data with metadata
    test_data = [
        ("data_group_1.csv",),
        ("data_group_2.csv",),
        ("data_group_3.csv",),
    ]
    
    test_df = spark.createDataFrame(test_data, ["_file_name"])
    
    # Apply file group extraction logic
    result_df = test_df.withColumn(
        "_file_group",
        F.regexp_extract(F.col("_file_name"), r"data_group_(\d+)\.csv", 1)
    )
    
    # Assert extraction - use & for AND, not &&
    count_1 = result_df.filter(
        (F.col("_file_name") == "data_group_1.csv") & 
        (F.col("_file_group") == "1")
    ).count()
    
    count_2 = result_df.filter(
        (F.col("_file_name") == "data_group_2.csv") & 
        (F.col("_file_group") == "2")
    ).count()
    
    count_3 = result_df.filter(
        (F.col("_file_name") == "data_group_3.csv") & 
        (F.col("_file_group") == "3")
    ).count()
    
    assert count_1 == 1, "Should extract file group 1"
    assert count_2 == 1, "Should extract file group 2"
    assert count_3 == 1, "Should extract file group 3"
    
    logger.info("test_read_csv_files_file_group_extraction passed")


In [0]:
def test_deduplicate_data_removes_duplicates():
    """Test that deduplicate_data removes duplicate records."""
    logger.info("Running test: deduplicate_data_removes_duplicates")
    
    # Create test data with duplicates
    test_schema = StructType([
        StructField("turbine_id", IntegerType(), False),
        StructField("timestamp", TimestampType(), False),
        StructField("wind_speed", DoubleType(), True),
        StructField("wind_direction", IntegerType(), True),
        StructField("power_output", DoubleType(), True),
        StructField("_ingested_at", TimestampType(), False),
        StructField("_source_file", StringType(), False),
    ])
    
    test_data = [
        # Turbine 1, timestamp 1 - 2 records (duplicates)
        (1, datetime(2022, 3, 1, 0, 0, 0), 9.1, 269, 2.9, datetime(2026, 8, 19, 10, 0, 0), "file1.csv"),
        (1, datetime(2022, 3, 1, 0, 0, 0), 9.1, 269, 3.1, datetime(2026, 8, 19, 11, 0, 0), "file1.csv"),
        
        # Turbine 1, timestamp 2 - 1 record
        (1, datetime(2022, 3, 1, 1, 0, 0), 9.2, 270, 3.0, datetime(2026, 8, 19, 10, 0, 0), "file1.csv"),
        
        # Turbine 2, timestamp 1 - 1 record
        (2, datetime(2022, 3, 1, 0, 0, 0), 11.3, 316, 2.5, datetime(2026, 8, 19, 10, 0, 0), "file2.csv"),
    ]
    
    test_df = create_test_dataframe(test_data, test_schema)
    
    # Apply deduplication
    result_df = deduplicate_data(test_df)
    
    # Assert: should have 3 records (1 duplicate removed)
    assert result_df.count() == 3, f"Expected 3 records, got {result_df.count()}"
    
    # Assert: for turbine 1, timestamp 1, should keep the latest (power_output=3.1)
    latest_record = result_df.filter(
        (F.col("turbine_id") == 1) & 
        (F.col("timestamp") == datetime(2022, 3, 1, 0, 0, 0))
    ).first()
    
    assert latest_record["power_output"] == 3.1, "Should keep latest record"
    
    logger.info("test_deduplicate_data_removes_duplicates passed")


def test_deduplicate_data_keeps_unique_records():
    """Test that deduplicate_data keeps all unique records."""
    logger.info("Running test: deduplicate_data_keeps_unique_records")
    
    test_schema = StructType([
        StructField("turbine_id", IntegerType(), False),
        StructField("timestamp", TimestampType(), False),
        StructField("wind_speed", DoubleType(), True),
        StructField("wind_direction", IntegerType(), True),
        StructField("power_output", DoubleType(), True),
        StructField("_ingested_at", TimestampType(), False),
    ])
    
    # All unique records
    test_data = [
        (1, datetime(2022, 3, 1, 0, 0, 0), 9.1, 269, 2.9, datetime(2026, 8, 19, 10, 0, 0)),
        (1, datetime(2022, 3, 1, 1, 0, 0), 9.2, 270, 3.0, datetime(2026, 8, 19, 10, 0, 0)),
        (2, datetime(2022, 3, 1, 0, 0, 0), 11.3, 316, 2.5, datetime(2026, 8, 19, 10, 0, 0)),
    ]
    
    test_df = create_test_dataframe(test_data, test_schema)
    result_df = deduplicate_data(test_df)
    
    # Assert: all records should be kept
    assert result_df.count() == 3, "Should keep all unique records"
    
    logger.info("test_deduplicate_data_keeps_unique_records passed")


In [0]:
def test_merge_into_bronze_creates_table():
    """Test that merge_into_bronze creates table if it doesn't exist."""
    logger.info("Running test: merge_into_bronze_creates_table")
    
    # Create test data
    test_schema = StructType([
        StructField("turbine_id", IntegerType(), False),
        StructField("timestamp", TimestampType(), False),
        StructField("wind_speed", DoubleType(), True),
        StructField("wind_direction", IntegerType(), True),
        StructField("power_output", DoubleType(), True),
        StructField("_source_file", StringType(), False),
        StructField("_file_name", StringType(), False),
        StructField("_file_group", StringType(), False),
        StructField("_ingested_at", TimestampType(), False),
        StructField("_ingestion_date", DateType(), False),
    ])
    
    test_data = [
        (1, datetime(2022, 3, 1, 0, 0, 0), 9.1, 269, 2.9, "file1.csv", "file1.csv", "1", 
         datetime(2026, 8, 19, 10, 0, 0), datetime(2026, 8, 19).date()),
    ]
    
    test_df = create_test_dataframe(test_data, test_schema)
    
    # Use a test table name
    test_table = "interviews_dev.bronze.turbine_raw_test"
    
    # Drop table if exists
    spark.sql(f"DROP TABLE IF EXISTS {test_table}")
    
    # Run merge
    result = merge_into_bronze(test_df, test_table)
    
    # Assert: table should be created
    assert result == "created", "Should create new table"
    assert spark.catalog.tableExists(test_table), "Table should exist"
    
    # Assert: table should have 1 record
    count = spark.table(test_table).count()
    assert count == 1, f"Table should have 1 record, got {count}"
    
    # Cleanup
    spark.sql(f"DROP TABLE IF EXISTS {test_table}")
    
    logger.info("test_merge_into_bronze_creates_table passed")


def test_merge_into_bronze_inserts_new_records():
    """Test that merge_into_bronze inserts new records."""
    logger.info("Running test: merge_into_bronze_inserts_new_records")
    
    test_schema = StructType([
        StructField("turbine_id", IntegerType(), False),
        StructField("timestamp", TimestampType(), False),
        StructField("wind_speed", DoubleType(), True),
        StructField("wind_direction", IntegerType(), True),
        StructField("power_output", DoubleType(), True),
        StructField("_source_file", StringType(), False),
        StructField("_file_name", StringType(), False),
        StructField("_file_group", StringType(), False),
        StructField("_ingested_at", TimestampType(), False),
        StructField("_ingestion_date", DateType(), False),
        StructField("_rescued_data", StringType(), True),  # ← ADD THIS
    ])
    
    # Create initial data
    initial_data = [
        (1, datetime(2022, 3, 1, 0, 0, 0), 9.1, 269, 2.9, "file1.csv", "file1.csv", "1", 
         datetime(2026, 8, 19, 10, 0, 0), datetime(2026, 8, 19).date(), None),  # ← ADD None
    ]
    
    initial_df = create_test_dataframe(initial_data, test_schema)
    
    # Create new data with additional record
    new_data = [
        (1, datetime(2022, 3, 1, 0, 0, 0), 9.1, 269, 2.9, "file1.csv", "file1.csv", "1", 
         datetime(2026, 8, 19, 10, 0, 0), datetime(2026, 8, 19).date(), None),  # ← ADD None
        (2, datetime(2022, 3, 1, 0, 0, 0), 11.3, 316, 2.5, "file2.csv", "file2.csv", "2", 
         datetime(2026, 8, 19, 10, 0, 0), datetime(2026, 8, 19).date(), None),  # ← ADD None
    ]
    
    new_df = create_test_dataframe(new_data, test_schema)
    
    # ... rest of test stays the same

def test_merge_into_bronze_updates_existing_records():
    """Test that merge_into_bronze updates existing records."""
    logger.info("Running test: merge_into_bronze_updates_existing_records")
    
    test_schema = StructType([
        StructField("turbine_id", IntegerType(), False),
        StructField("timestamp", TimestampType(), False),
        StructField("wind_speed", DoubleType(), True),
        StructField("wind_direction", IntegerType(), True),
        StructField("power_output", DoubleType(), True),
        StructField("_source_file", StringType(), False),
        StructField("_file_name", StringType(), False),
        StructField("_file_group", StringType(), False),
        StructField("_ingested_at", TimestampType(), False),
        StructField("_ingestion_date", DateType(), False),
        StructField("_rescued_data", StringType(), True),  # ← ADD THIS
    ])
    
    # Create initial data
    initial_data = [
        (1, datetime(2022, 3, 1, 0, 0, 0), 9.1, 269, 2.9, "file1.csv", "file1.csv", "1", 
         datetime(2026, 8, 19, 10, 0, 0), datetime(2026, 8, 19).date(), None),  # ← ADD None
    ]
    
    # ... rest of test stays the same

In [0]:
def test_calculate_data_quality_metrics():
    """Test that calculate_data_quality_metrics runs without errors."""
    logger.info("Running test: calculate_data_quality_metrics")
    
    # Create test table
    test_schema = StructType([
        StructField("turbine_id", IntegerType(), False),
        StructField("timestamp", TimestampType(), False),
        StructField("wind_speed", DoubleType(), True),
        StructField("wind_direction", IntegerType(), True),
        StructField("power_output", DoubleType(), True),
    ])
    
    test_data = [
        (1, datetime(2022, 3, 1, 0, 0, 0), 9.1, 269, 2.9),
        (2, datetime(2022, 3, 1, 0, 0, 0), 11.3, 316, 2.5),
    ]
    
    test_df = create_test_dataframe(test_data, test_schema)
    test_table = "interviews_dev.bronze.turbine_raw_test"
    
    # Drop table if exists
    spark.sql(f"DROP TABLE IF EXISTS {test_table}")
    
    # Create table
    test_df.write.format("delta").mode("overwrite").saveAsTable(test_table)
    
    # Run metrics calculation
    try:
        calculate_data_quality_metrics(test_table)
        logger.info("test_calculate_data_quality_metrics passed")
    except Exception as e:
        logger.error(f"test_calculate_data_quality_metrics failed: {str(e)}")
        raise
    finally:
        # Cleanup
        spark.sql(f"DROP TABLE IF EXISTS {test_table}")

In [0]:
def test_display_sample_data():
    """Test that display_sample_data runs without errors."""
    logger.info("Running test: display_sample_data")
    
    # Create test table
    test_schema = StructType([
        StructField("turbine_id", IntegerType(), False),
        StructField("timestamp", TimestampType(), False),
        StructField("wind_speed", DoubleType(), True),
        StructField("wind_direction", IntegerType(), True),
        StructField("power_output", DoubleType(), True),
        StructField("_ingestion_date", DateType(), False),
    ])
    
    test_data = [
        (1, datetime(2022, 3, 1, 0, 0, 0), 9.1, 269, 2.9, datetime(2026, 8, 19).date()),
        (2, datetime(2022, 3, 1, 0, 0, 0), 11.3, 316, 2.5, datetime(2026, 8, 19).date()),
    ]
    
    test_df = create_test_dataframe(test_data, test_schema)
    test_table = "interviews_dev.bronze.turbine_raw_test"
    
    # Drop table if exists
    spark.sql(f"DROP TABLE IF EXISTS {test_table}")
    
    # Create table
    test_df.write.format("delta").mode("overwrite").saveAsTable(test_table)
    
    # Run display
    try:
        display_sample_data(test_table)
        logger.info("test_display_sample_data passed")
    except Exception as e:
        logger.error(f"test_display_sample_data failed: {str(e)}")
        raise
    finally:
        # Cleanup
        spark.sql(f"DROP TABLE IF EXISTS {test_table}")

# Execute Unit Tests

In [0]:
def run_all_tests():
    """Run all unit tests."""
    logger.info("=" * 80)
    logger.info("Starting Unit Tests for Bronze Layer")
    logger.info("=" * 80)
    
    tests = [
        test_read_csv_files_structure,
        test_read_csv_files_file_group_extraction,
        test_deduplicate_data_removes_duplicates,
        test_deduplicate_data_keeps_unique_records,
        test_merge_into_bronze_creates_table,
        test_merge_into_bronze_inserts_new_records,
        test_merge_into_bronze_updates_existing_records,
        test_calculate_data_quality_metrics,
        test_display_sample_data,
    ]
    
    passed = 0
    failed = 0
    
    for test_func in tests:
        try:
            test_func()
            passed += 1
        except Exception as e:
            logger.error(f"{test_func.__name__} failed: {str(e)}")
            failed += 1
    
    logger.info(f"Test Results: {passed} passed, {failed} failed")
    if failed > 0:
        raise AssertionError(f"{failed} test(s) failed")
    
    logger.info("All tests passed!")


# Run tests
run_all_tests()